In [15]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Load food data
food_data = pd.read_csv('resisDataMP.csv')

# Preprocess food data
def preprocess_data(df):
    df['calories'] = df['energi_kal']
    df['carbs'] = df['karbohidrat_gram']
    df['protein'] = df['protein_gram']
    df['fats'] = df['lemak_gram']
    return df

food_data = preprocess_data(food_data)

# Calculate BMR using Harris-Benedict formula
def calculate_bmr(weight, height, age, sex):
    if sex == 'female':
        return 447.593 + 9.247 * weight + 3.098 * height - 4.330 * age
    else:
        return 88.362 + 13.397 * weight + 4.799 * height - 5.677 * age

# Calculate Total Energy Expenditure (TEE)
def calculate_tee(bmr, activity_level, trimester):
    activity_multiplier = {
        'sedentary': 1.2,
        'lightly_active': 1.375,
        'moderately_active': 1.55,
        'very_active': 1.725,
        'extra_active': 1.9
    }
    
    trimester_multiplier = {
        1: 1.0,
        2: 1.1,
        3: 1.3
    }
    
    return bmr * activity_multiplier[activity_level] * trimester_multiplier[trimester]

# Example user data
weight = 70  # kg
height = 165  # cm
age = 30  # years
sex = 'female'
activity_level = 'moderately_active'
trimester = 2

bmr = calculate_bmr(weight, height, age, sex)
tee = calculate_tee(bmr, activity_level, trimester)
daily_calories = tee

# Isi Piringku guidelines (one plate = 700 calories)
plate_calories = 700
staple_food_calories = (2/6) * plate_calories
side_dish_calories = (1/6) * plate_calories
fruit_calories = (1/6) * plate_calories
vegetable_calories = (2/6) * plate_calories

# Filter food data according to Isi Piringku guidelines
def filter_food_data(df):
    staple_foods = df[df['jenis_pangan'] == 'mp']
    side_dishes = df[df['jenis_pangan'].isin(['ikan kerang udang', 'daging unggas', 'kacang biji bean'])]
    fruits = df[df['jenis_pangan'] == 'buah']
    vegetables = df[df['jenis_pangan'] == 'sayuran']
    return staple_foods, side_dishes, fruits, vegetables

staple_foods, side_dishes, fruits, vegetables = filter_food_data(food_data)

# KNN algorithm to generate meal recommendations
def knn_recommendation(staple_foods, side_dishes, fruits, vegetables, k=10):
    features = ['calories', 'carbs', 'protein', 'fats']
    
    knn_staple = NearestNeighbors(n_neighbors=k).fit(staple_foods[features])
    knn_side = NearestNeighbors(n_neighbors=k).fit(side_dishes[features])
    knn_fruit = NearestNeighbors(n_neighbors=k).fit(fruits[features])
    knn_vegetable = NearestNeighbors(n_neighbors=k).fit(vegetables[features])
    
    _, indices_staple = knn_staple.kneighbors(pd.DataFrame([[staple_food_calories, 0.6 * plate_calories / 4, 0, 0]], columns=features))
    _, indices_side = knn_side.kneighbors(pd.DataFrame([[side_dish_calories, 0, 0.15 * plate_calories / 4, 0]], columns=features))
    _, indices_fruit = knn_fruit.kneighbors(pd.DataFrame([[fruit_calories, 0, 0, 0]], columns=features))
    _, indices_vegetable = knn_vegetable.kneighbors(pd.DataFrame([[vegetable_calories, 0, 0, 0]], columns=features))
    
    staple_options = staple_foods.iloc[indices_staple[0]]
    side_options = side_dishes.iloc[indices_side[0]]
    fruit_options = fruits.iloc[indices_fruit[0]]
    vegetable_options = vegetables.iloc[indices_vegetable[0]]
    
    return staple_options, side_options, fruit_options, vegetable_options

# Generate meal recommendations
def generate_random_plate(staple_options, side_options, fruit_options, vegetable_options):
    staple = staple_options.sample()
    side = side_options.sample()
    fruit = fruit_options.sample()
    vegetable = vegetable_options.sample()
    
    plate = [staple['nama_bahan'].values[0], side['nama_bahan'].values[0], 
             fruit['nama_bahan'].values[0], vegetable['nama_bahan'].values[0]]
    
    total_calories = staple['calories'].values[0] + side['calories'].values[0] + fruit['calories'].values[0] + vegetable['calories'].values[0]
    total_carbs = staple['carbs'].values[0] + side['carbs'].values[0] + fruit['carbs'].values[0] + vegetable['carbs'].values[0]
    total_protein = staple['protein'].values[0] + side['protein'].values[0] + fruit['protein'].values[0] + vegetable['protein'].values[0]
    total_fats = staple['fats'].values[0] + side['fats'].values[0] + fruit['fats'].values[0] + vegetable['fats'].values[0]
    
    return {
        'plate': plate,
        'total_calories': total_calories,
        'total_carbs': total_carbs,
        'total_protein': total_protein,
        'total_fats': total_fats
    }

# Generate meal recommendations
staple_options, side_options, fruit_options, vegetable_options = knn_recommendation(staple_foods, side_dishes, fruits, vegetables)

# Generate multiple random plates
num_recommendations = 5
recommended_plates = [generate_random_plate(staple_options, side_options, fruit_options, vegetable_options) for _ in range(num_recommendations)]

# Print recommendations
for i, plate in enumerate(recommended_plates):
    print(f"Plate {i+1}: {plate['plate']}")
    print(f"Total Calories: {plate['total_calories']:.2f} kcal")
    print(f"Total Carbs: {plate['total_carbs']:.2f} g")
    print(f"Total Protein: {plate['total_protein']:.2f} g")
    print(f"Total Fats: {plate['total_fats']:.2f} g")
    print()

Plate 1: ['mie telur lebar keriting cap ayam 2 telor', 'ikan ekor kuning segar', 'pisang ambon segar', 'Tumis Kacang Panjang']
Total Calories: 700.00 kcal
Total Carbs: 87.87 g
Total Protein: 61.42 g
Total Fats: 15.31 g

Plate 2: ['ubi cilembu', 'ikan mujahir pepes', 'Buavita Apple', 'Tumis Timun dan Ayam']
Total Calories: 623.00 kcal
Total Carbs: 79.03 g
Total Protein: 33.76 g
Total Fats: 20.43 g

Plate 3: ['SUPERIOR Bihun Jagung urai', 'Stim Ikan Ala Chinese', 'pisang talas segar', 'Sawi Tahu']
Total Calories: 694.00 kcal
Total Carbs: 114.44 g
Total Protein: 44.16 g
Total Fats: 12.66 g

Plate 4: ['ubi cilembu', 'ikan mujahir pepes', 'sawo kecik segar', 'andewi segar']
Total Calories: 644.00 kcal
Total Carbs: 72.00 g
Total Protein: 26.10 g
Total Fats: 5.50 g

Plate 5: ['mie telur lebar keriting cap ayam 2 telor', 'ikan lemuru segar', 'kawista segar', 'Sawi Tahu']
Total Calories: 686.00 kcal
Total Carbs: 92.40 g
Total Protein: 49.97 g
Total Fats: 16.69 g

